# Voorbeelduitwerkingen Bot v2 en Bot v3 — *docentmateriaal*

> **Niet uitdelen aan studenten.** Dit notebook staat er voor jou: welke programmeerconcepten je per botversie mag verwachten, en één uitgewerkte oplossing per versie om tegenaan te houden bij het nakijken en bij vragen aan de balie. De code staat ook los naast dit notebook, als `_docent_bot_v2.py` en `_docent_bot_v3.py`.

Beide bots hieronder zijn door de echte validator gehaald (`valideer_bot_code`, week 3 en week 5) en hebben een echt toernooi gespeeld. Ze zijn expres *goed maar niet optimaal*: ze laten zien hoe de concepten samenkomen, niet hoe je het toernooi wint.

**Eén ding om vooraf te weten**, want hier lopen studenten op vast:

> `schat_winkans` uit `_hulpfuncties_week3.py` is op de server **niet beschikbaar**. Een ingeleverde bot die die functie importeert, wordt afgekeurd met `ModuleNotFoundError`. De winkansen moeten dus als dictionary in de bot zelf staan — één keer opzoeken in het notebook, dan de getallen overnemen. In Werkcollege 4 Deel 6 staat dat nu expliciet, met een cel die de dictionary kant-en-klaar print.

---

## Deel 1 — Welke programmeerconcepten zitten er in Bot v2?

Alles wat een student voor v2 nodig heeft, is in Week 1 en Werkcollege 4 behandeld. Er zit niets nieuws in — de moeilijkheid zit in het combineren.

| Concept | Waar geleerd | Hoe het in de bot terugkomt |
|---|---|---|
| Functie met parameters en `return` | WC1 Deel 1 | `kies_actie` moet een string teruggeven, niet printen |
| Default-argumenten | WC4 Deel 1 | `ronde="preflop"`, `pot=0` — anders werkt de bot niet als de engine ze niet meegeeft |
| `if` / `elif` / `else` | WC1 Deel 1 | de beslisboom zelf; volgorde is hier het hele verhaal |
| Vergelijkingsoperatoren | WC1 Deel 1 | `stack < 150`, `winkans >= 70` |
| Lijst-indexering | WC1 Deel 1 | `hand[0]`, `hand[1]` |
| Dictionary als opzoektabel | WC4 Deel 1 | winkansen én de marge per straat |
| `.get()` met een default | WC4 Deel 1 + oefenstof | een hand of ronde die niet in de tabel staat mag geen `KeyError` geven |
| Ternary (`a if x else b`) | WC3 Ronde 0 Deel A | `return "raise" if winkans >= 70 else "call"` |
| `sorted()` + `"".join()` | WC3 Ronde 0 Deel A | van `['K','A']` naar één sleutel, zodat de volgorde niet uitmaakt |
| Rekenen met percentages | WC4 Deel 6 | de pot odds: `inzet / (pot + inzet) * 100` |

**Waar het bij nakijken meestal misgaat**

1. **`print` in plaats van `return`.** De bot lijkt te werken in het notebook, maar de engine krijgt `None` en de bot foldt elke hand.
2. **Geen default op `ronde` of `pot`.** Werkt in de test, crasht in het toernooi.
3. **`WINKANS[sleutel]` in plaats van `WINKANS.get(sleutel, ...)`.** Eén onbekende hand en de bot crasht — dan foldt hij die hand en draait het toernooi door, dus het valt niet op.
4. **De volgorde van de `if`-takken.** Wie `stack < 150` onderaan zet, komt daar nooit; net als de onbereikbare `elif` uit de oefenstof.

---

## Deel 2 — Uitgewerkte Bot v2

Deze versie gebruikt elk concept uit de tabel precies één keer, en niets meer. De vier `if`-takken zijn bewust in deze volgorde gezet: van "geen keuze meer" naar "vrije keuze".

1. **Korte stack** (< 150) — met zo weinig chips heeft manoeuvreren geen zin: alles of niets.
2. **Gratis meedoen** (`inzet_om_te_callen == 0`) — nooit folden als het niets kost.
3. **Te weinig winkans voor de prijs** — de pot-odds-regel uit Deel 6, plus een marge die per straat strenger wordt.
4. **Anders meedoen**, en raisen als de hand echt goed is.

In [ ]:
# --- Bot v2: opzoektabel + pot odds, met een marge die per straat strenger wordt ---
WINKANS = {
    "AA": 85.7, "KK": 82.8, "QQ": 79.9, "JJ": 77.5,
    "KA": 65.7, "QA": 65.4, "QK": 63.4, "QJ": 59.6, "910": 53.8, "72": 34.6,
}
MARGE_PER_RONDE = {"preflop": 0, "flop": 5, "turn": 10, "river": 15}


def hand_sleutel(hand):
    """['K', 'A'] en ['A', 'K'] geven dezelfde sleutel, dus de volgorde maakt niet uit."""
    return "".join(sorted(hand, reverse=True))


def kies_actie(hand, stack, ronde="preflop", pot=0, inzet_om_te_callen=0):
    winkans = WINKANS.get(hand_sleutel(hand), 40)
    marge = MARGE_PER_RONDE.get(ronde, 10)

    if stack < 150:
        return "all_in" if winkans >= 60 else "fold"

    if inzet_om_te_callen == 0:
        return "raise" if winkans >= 70 else "call"

    vereiste_winkans = inzet_om_te_callen / (pot + inzet_om_te_callen) * 100
    if winkans < vereiste_winkans + marge:
        return "fold"
    return "raise" if winkans >= 70 else "call"

### De marge per straat is de kern van deze bot

De pot-odds-regel alleen is naïef: die zegt bij een kleine inzet bijna altijd "call", ook met `7 2`. De marge maakt hem strenger naarmate er meer geld in de pot zit en er minder onbekend is:

- **preflop** (marge 0): puur de wiskunde, want je weet nog niets van het bord.
- **river** (marge 15): alle vijf kaarten liggen er. Je opzoektabel gaat over je *starthand* en zegt niets over wat het bord met je hand deed — dus eis je een flinke buffer boven wat de pot odds minimaal vragen.

Dat is ook precies het gesprek dat je met studenten wil hebben: hun tabel is preflop-informatie, en ze gebruiken 'm op vier straten.

In [ ]:
import sys
sys.path.insert(0, ".")
from _docent_bot_v2 import kies_actie as bot_v2

print("situatie                                   -> actie")
for hand, stack, ronde, pot, inzet in [
    (["A", "K"], 1000, "preflop", 100, 20),
    (["A", "K"], 1000, "river",   100, 20),
    (["7", "2"], 1000, "preflop", 100, 20),
    (["7", "2"], 1000, "preflop", 100, 300),
    (["A", "A"],   80, "preflop", 100, 20),
    (["7", "2"],   80, "preflop", 100, 20),
    (["Q", "J"], 1000, "flop",      0,   0),
]:
    situatie = f"{str(hand):<12} stack {stack:>4} {ronde:<8} pot {pot:>3} bijleggen {inzet:>3}"
    print(f"{situatie} -> {bot_v2(hand, stack, ronde=ronde, pot=pot, inzet_om_te_callen=inzet)}")

---

## Deel 3 — Welke concepten komen er in Bot v3 bij?

v3 is v2 plus drie dingen. De Python die daarvoor nodig is, is één stap zwaarder:

| Concept | Waar geleerd | Hoe het in de bot terugkomt |
|---|---|---|
| Geneste dictionary | WC7 Deel 2 | `MARGE[strategie][ronde]` — een tabel met twee ingangen |
| `random.random() < kans` | WC7 Deel 2 | bluffen als frequentie in plaats van als vaste keuze |
| `for`-loop over een lijst met dicts | WC7 Deel 3 | `tegenstander_acties_deze_hand` doorlopen |
| Een eigen hulpfunctie die `True`/`False` teruggeeft | WC1 Deel 2 | `iemand_heeft_geraised(acties)` — leesbaarder dan de lus in de beslisboom |
| `or []` als vangnet | WC7 Deel 3 | de lijst kan `None` zijn; dan moet de lus niet crashen |
| Een variabele bijwerken op basis van een voorwaarde | WC1 Deel 1 | `marge += 5` als de tafel kracht toonde |

**En één inhoudelijk punt dat geen Python is.** Vanaf v3 speelt de bot niet tegen kaarten maar tegen 43 andere programma's, en dat verandert wat goed spel is. Een bot is voorspelbaar, leest jou niet, en is meestal te tight. Daarom zit `bluf_kans` erin: een vaste regel is uitbuitbaar, een kans is dat niet. Dat staat als opdracht in Werkcollege 7 Deel 4.

---

## Deel 4 — Uitgewerkte Bot v3

Drie dingen zijn nieuw ten opzichte van v2:

1. **`MARGE[strategie][ronde]`** — vier archetypes met elk hun eigen strengheid per straat. `tight` eist overal een dikke buffer, `loose` doet mee met minder. Dít is waarom strategie pas vanaf Week 5 zin heeft: in Week 3 had de bot niets om het label op toe te passen.
2. **Bluffen met een kans**, en alleen als de tafel nog geen kracht heeft getoond. Bluffen tegen iemand die net raiste is weggegooid geld.
3. **`tegenstander_acties_deze_hand`** — één raise van een tegenstander maakt deze bot 5 punten strenger.

In [ ]:
# --- Bot v3: v2 + strategie, bluffen, en kijken wat de tafel deed ---
import random

WINKANS = {
    "AA": 85.7, "KK": 82.8, "QQ": 79.9, "JJ": 77.5,
    "KA": 65.7, "QA": 65.4, "QK": 63.4, "QJ": 59.6, "910": 53.8, "72": 34.6,
}
MARGE = {
    "tight":      {"preflop": 10, "flop": 12, "turn": 15, "river": 20},
    "loose":      {"preflop": -5, "flop": 0,  "turn": 3,  "river": 5},
    "balanced":   {"preflop": 0,  "flop": 5,  "turn": 8,  "river": 10},
    "aggressive": {"preflop": -3, "flop": 2,  "turn": 4,  "river": 6},
}
RAISE_GRENS = {"tight": 75, "loose": 60, "balanced": 68, "aggressive": 55}


def hand_sleutel(hand):
    return "".join(sorted(hand, reverse=True))


def iemand_heeft_geraised(acties):
    """acties is een lijst met dicts: {'bot_naam': ..., 'actie': ..., 'bedrag': ...}."""
    for actie in acties or []:
        if actie.get("actie") == "raise":
            return True
    return False


def kies_actie(hand, stack, strategie, bluf_kans, ronde="preflop", pot=0,
               inzet_om_te_callen=0, tegenstander_acties_deze_hand=None):
    winkans = WINKANS.get(hand_sleutel(hand), 40)
    marge = MARGE.get(strategie, MARGE["balanced"]).get(ronde, 10)
    raise_grens = RAISE_GRENS.get(strategie, 68)
    tafel_toonde_kracht = iemand_heeft_geraised(tegenstander_acties_deze_hand)

    if stack < 150:
        return "all_in" if winkans >= 60 else "fold"

    # bluffen: alleen als niemand al kracht heeft getoond, anders is het weggegooid geld
    if not tafel_toonde_kracht and winkans < 45 and random.random() < bluf_kans:
        return "raise"

    if inzet_om_te_callen == 0:
        return "raise" if winkans >= raise_grens else "call"

    vereiste_winkans = inzet_om_te_callen / (pot + inzet_om_te_callen) * 100
    if tafel_toonde_kracht:
        marge += 5   # iemand heeft geraised: wees strenger
    if winkans < vereiste_winkans + marge:
        return "fold"
    return "raise" if winkans >= raise_grens else "call"

In [ ]:
import random
from _docent_bot_v3 import kies_actie as bot_v3

geen = []
er_werd_geraised = [{"bot_naam": "500123__w5", "actie": "raise", "bedrag": 60}]

print("A. De marge beslist  (10 9, flop, pot 100, bijleggen 100 -> je hebt 50% nodig, je hebt 53,8%)")
for strategie in ["tight", "loose", "balanced", "aggressive"]:
    rustig = bot_v3(["10", "9"], 1000, strategie, 0.0, ronde="flop", pot=100,
                    inzet_om_te_callen=100, tegenstander_acties_deze_hand=geen)
    na_raise = bot_v3(["10", "9"], 1000, strategie, 0.0, ronde="flop", pot=100,
                      inzet_om_te_callen=100, tegenstander_acties_deze_hand=er_werd_geraised)
    print(f"   {strategie:<11} rustige tafel: {rustig:<6} na een raise: {na_raise}")

print("\nB. De raise-grens beslist  (A K, flop, gratis meedoen)")
for strategie in ["tight", "loose", "balanced", "aggressive"]:
    actie = bot_v3(["A", "K"], 1000, strategie, 0.0, ronde="flop", pot=100,
                   inzet_om_te_callen=0, tegenstander_acties_deze_hand=geen)
    print(f"   {strategie:<11} {actie}")

print("\nC. Bluffen is een frequentie, geen keuze  (7 2, bluf_kans 0.3, 200 handen)")
for acties, label in [(geen, "rustige tafel"), (er_werd_geraised, "na een raise")]:
    random.seed(1)
    keer = sum(
        bot_v3(["7", "2"], 1000, "balanced", 0.3, ronde="river", pot=100,
               inzet_om_te_callen=40, tegenstander_acties_deze_hand=acties) == "raise"
        for _ in range(200)
    )
    print(f"   {label:<15} {keer}/200 keer een bluf-raise")

### Wat deze test laat zien

**A — de marge beslist.** Bij een hand die net op het randje zit (53,8% winkans, 50% nodig) folden `tight` en `balanced`, terwijl `loose` en `aggressive` meedoen. En zodra er iemand geraised heeft, folden ze alle vier: die `marge += 5` duwt precies die twee over de streep. Dít is het gedrag dat je bij een student wil zien — vier labels die op dezelfde hand verschillend beslissen.

**B — de raise-grens beslist.** Met `A K` en gratis meedoen raisen `loose` en `aggressive`, en checken `tight` en `balanced` mee. Twee tabellen (`MARGE` en `RAISE_GRENS`) geven samen vier verschillende speelstijlen, zonder dat de beslisboom ingewikkelder wordt.

**C — bluffen is een frequentie.** Op een rustige tafel bluft deze bot 62 van de 200 keer: ongeveer 31%, en dat is de `bluf_kans` van 0,3 die je terugziet. Na een raise van een tegenstander bluft hij 0 van de 200 keer, want die tak wordt dan overgeslagen. Dat is het hele punt van bluffen met een kans in plaats van met een regel: er is geen enkele hand waaruit een tegenstander kan aflezen wat je hebt.

**Handige controle bij nakijken:** laat de student zijn bot met alle vier de strategieën op hetzelfde setje situaties draaien en de uitkomsten naast elkaar zetten. Zijn ze identiek, dan is `strategie` decoratie. Dat is de snelste manier om te zien of hij het label echt gebruikt.

---

## Deel 5 — Bewijs: door de validator en door een echt toernooi

Hieronder draaien beide bots door dezelfde validatie als een echte inzending, en spelen ze een toernooi tegen elkaar. Dit is ook het snelste antwoord op "maar mijn bot werkt wél" — laat 'm hier zien.

In [ ]:
sys.path.insert(0, "../api")
from bot_validator import valideer_bot_code

for pad, week, extra in [("_docent_bot_v2.py", 3, {}),
                         ("_docent_bot_v3.py", 5, {"strategie": "tight", "bluf_kans": 0.2})]:
    code = open(pad).read()
    resultaat = valideer_bot_code(code, week=week, **extra)
    print(f"{pad} (week {week}): geldig = {resultaat['geldig']}")
    if resultaat["foutmelding"]:
        print("   ", resultaat["foutmelding"][:200])
    if resultaat.get("actie_resultaten"):
        print("    acties die eruit kwamen:", sorted(set(resultaat["actie_resultaten"])))

In [ ]:
from poker_adapter import speel_toernooi

bots = {
    "v2":            {"kies_actie": bot_v2},
    "v3-tight":      {"kies_actie": bot_v3, "strategie": "tight",      "bluf_kans": 0.1},
    "v3-loose":      {"kies_actie": bot_v3, "strategie": "loose",      "bluf_kans": 0.3},
    "v3-aggressive": {"kies_actie": bot_v3, "strategie": "aggressive", "bluf_kans": 0.4},
}

uitkomst = speel_toernooi(bots, n_simulaties=3, n_handen=30, seed=7)
for naam, stand in sorted(uitkomst["eindstand_per_bot"].items(), key=lambda p: -p[1]):
    print(f"{naam:<16} {stand}")

---

## Deel 6 — Nakijkchecklist

Voor beide versies, in de volgorde waarin het meestal misgaat:

| Check | Waarom |
|---|---|
| Geeft `kies_actie` een string **terug** (geen `print`)? | anders krijgt de engine `None` en foldt de bot elke hand |
| Hebben alle optionele parameters een default? | zonder default crasht de bot zodra de engine ze niet meegeeft |
| Wordt er nergens `_hulpfuncties_week3` geïmporteerd? | dat bestand staat niet op de server |
| Staat er een `.get()` met default op elke dictionary-lookup? | een onbekende hand mag geen `KeyError` geven |
| Is elke `if`-tak bereikbaar? | dezelfde fout als de onbereikbare `elif` uit de oefenstof |
| Gebruikt v3 zijn strategie écht? | vier archetypes op hetzelfde setje situaties moeten verschillende uitkomsten geven |
| Gebruikt v3 `tegenstander_acties_deze_hand` met een vangnet voor `None`? | de lijst is leeg bij de eerste beslissing van een hand |

Voor het huiswerk (4–8 uur, in Week 3 én Week 5) geldt: een bot die alle vinkjes haalt is *geldig*, niet *goed*. Waar je naar kijkt bij de beoordeling is of er onderbouwde keuzes in zitten — drempels die ergens op gebaseerd zijn, en een student die kan uitleggen waarom hij ze zo heeft gezet.